# ☕ Cafe Sales Data Cleaning & Analysis Project

## Step 1 | Importing the Libraries

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual themes
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120

## Step 2 | Loading the Raw Dataset

In [ ]:
# Load raw dataset
cafe_records = pd.read_csv("dirty_cafe_sales.csv")
cafe_records.head()

## Step 3 | Inspecting the Data Structure

In [ ]:
# Display dataset dimensions and summary info
print("Rows:", cafe_records.shape[0])
print("Cols:", cafe_records.shape[1])
cafe_records.info()

In [ ]:
# Count null values in each column
cafe_records.isnull().sum()

In [ ]:
# Count hidden invalid strings (ERROR and UNKNOWN)
bogus_labels = ["ERROR", "UNKNOWN"]
dirty_counts = cafe_records.apply(lambda col: col.isin(bogus_labels).sum())
dirty_counts[dirty_counts > 0]

## Step 4 | Replacing 'ERROR' and 'UNKNOWN' with NaNs

In [ ]:
# Replace dirty string placeholders with NaN
sales_ledger = cafe_records.replace(bogus_labels, np.nan)
sales_ledger.isnull().sum()

## Step 5 | Handling Missing Categorical Values

In [ ]:
# Impute missing categorical values using column mode
top_item = sales_ledger["Item"].mode().iloc[0]
sales_ledger["Item"] = sales_ledger["Item"].fillna(top_item)

top_payment = sales_ledger["Payment Method"].mode().iloc[0]
sales_ledger["Payment Method"] = sales_ledger["Payment Method"].fillna(top_payment)

top_location = sales_ledger["Location"].mode().iloc[0]
sales_ledger["Location"] = sales_ledger["Location"].fillna(top_location)

print(f"Item filled with: {top_item}")
print(f"Payment filled with: {top_payment}")
print(f"Location filled with: {top_location}")

## Step 6 | Handling Numeric Columns

In [ ]:
# Convert numeric columns and fill missing values with median
sales_ledger["Quantity"] = pd.to_numeric(sales_ledger["Quantity"], errors="coerce")
sales_ledger["Price Per Unit"] = pd.to_numeric(sales_ledger["Price Per Unit"], errors="coerce")
sales_ledger["Total Spent"] = pd.to_numeric(sales_ledger["Total Spent"], errors="coerce")

median_qty = sales_ledger["Quantity"].median()
median_price = sales_ledger["Price Per Unit"].median()

sales_ledger["Quantity"] = sales_ledger["Quantity"].fillna(median_qty)
sales_ledger["Price Per Unit"] = sales_ledger["Price Per Unit"].fillna(median_price)

print(f"Quantity median: {median_qty}")
print(f"Price median: {median_price}")

## Step 7 | Recalculating the Total Spent Column

In [ ]:
# Recalculate Total Spent = Quantity * Price Per Unit
sales_ledger["Total Spent"] = sales_ledger["Quantity"] * sales_ledger["Price Per Unit"]
sales_ledger[["Quantity", "Price Per Unit", "Total Spent"]].head(10)

## Step 8 | Parsing and Handling Transaction Date

In [ ]:
# Parse dates (invalid values become NaT)
sales_ledger["Transaction Date"] = pd.to_datetime(sales_ledger["Transaction Date"], errors="coerce")
missing_dates = sales_ledger["Transaction Date"].isna().sum()
print(f"Missing dates: {missing_dates}")

## Step 9 | Post-Cleaning Verification

In [ ]:
# Verify remaining null values
sales_ledger.isnull().sum()

In [ ]:
# View numeric columns summary statistics
sales_ledger.describe()

In [ ]:
# Show first 10 rows of cleaned dataset
sales_ledger.head(10)

## Step 10 | Exploratory Data Analysis & Visualizations

In [ ]:
# Group sales by Item
item_summary = (
    sales_ledger
    .groupby("Item", as_index=False)
    .agg(
        order_count  = ("Item", "size"),
        total_revenue = ("Total Spent", "sum"),
        avg_spend     = ("Total Spent", "mean")
    )
    .sort_values("total_revenue", ascending=False)
)
item_summary

In [ ]:
# Bar chart: Total revenue by item
plt.figure(figsize=(10, 5))
sns.barplot(
    data=item_summary,
    y="Item",
    x="total_revenue",
    palette="viridis",
)
plt.title("Total Revenue by Menu Item")
plt.xlabel("Revenue")
plt.ylabel("Item")
plt.tight_layout()
plt.show()

In [ ]:
# Pie chart: Payment method distribution
payment_counts = sales_ledger["Payment Method"].value_counts()

plt.figure(figsize=(7, 7))
plt.pie(
    payment_counts,
    labels=payment_counts.index,
    autopct="%1.1f%%",
    startangle=140,
    colors=sns.color_palette("pastel"),
)
plt.title("Payment Method Distribution")
plt.tight_layout()
plt.show()

In [ ]:
# Bar chart: Transactions count by location
location_counts = sales_ledger["Location"].value_counts()

plt.figure(figsize=(8, 5))
sns.barplot(
    x=location_counts.index,
    y=location_counts.values,
    palette="coolwarm",
)
plt.title("Transactions by Location")
plt.xlabel("Location")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
# Bar chart: Monthly revenue trend (2023)
dated_sales = sales_ledger.dropna(subset=["Transaction Date"]).copy()
dated_sales["month"] = dated_sales["Transaction Date"].dt.to_period("M")

monthly_revenue = dated_sales.groupby("month")["Total Spent"].sum()

plt.figure(figsize=(12, 5))
monthly_revenue.plot(kind="bar", color="steelblue", edgecolor="white")
plt.title("Monthly Revenue (2023)")
plt.xlabel("Month")
plt.ylabel("Revenue")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Histogram: Distribution of total spent per transaction
plt.figure(figsize=(10, 5))
sns.histplot(sales_ledger["Total Spent"], bins=25, kde=True, color="coral")
plt.title("Distribution of Total Spent per Transaction")
plt.xlabel("Total Spent")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

## Step 11 | Exporting the Cleaned Dataset

In [ ]:
# Save clean dataset back to CSV
sales_ledger.to_csv("cleaned_cafe_sales.csv", index=False)
print("✅ Cleaned data exported successfully")
print(f"   Shape: {sales_ledger.shape}")